### Import libraries

In [1]:
import pandas as pd
import numpy as np
import re
import joblib
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to C:\Users\Shrabani
[nltk_data]     P\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Shrabani
[nltk_data]     P\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Shrabani
[nltk_data]     P\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

### Load the processed sentiment dataset

In [3]:
df = pd.read_csv(r"C:\Users\Shrabani P\IronHack\Week6_Day2\Project_3\Project3_amazon-review-insights\data\processed\amazon_reviews_sentiment.csv")


df.head()

,name,brand,categories,reviews.rating,reviews.title,reviews.text,sentiment
0,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",5.0,Kindle,This product so far has not disappointed. My c...,Positive
1,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",5.0,very fast,great for beginner or experienced person. Boug...,Positive
2,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",5.0,Beginner tablet for our 9 year old son.,Inexpensive tablet for him to use and learn on...,Positive
3,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",4.0,Good!!!,I've had my Fire HD 8 two weeks now and I love...,Positive
4,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",5.0,Fantastic Tablet for kids,I bought this for my grand daughter when she c...,Positive


In [4]:
df.columns

Index(['name', 'brand', 'categories', 'reviews.rating', 'reviews.title',
       'reviews.text', 'sentiment'],
      dtype='object')

### Create the Preprocessing Function

In [5]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    if pd.isna(text):
        return ""

    text = text.lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    words = text.split()
    words = [word for word in words if word not in stop_words]
    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words)

### Clean the Reviews


In [6]:
df["clean_review"] = df["reviews.text"].apply(clean_text)

### Remove empty revies

In [7]:
df = df[df["clean_review"] != ""].reset_index(drop=True)

In [8]:
df.columns

Index(['name', 'brand', 'categories', 'reviews.rating', 'reviews.title',
       'reviews.text', 'sentiment', 'clean_review'],
      dtype='object')

### Define features and target

In [9]:
X = df["clean_review"]
y = df["sentiment"]

### Train-Validation - test split

In [10]:
# First split: 85% train+validation, 15% test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

In [11]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.1765,
    random_state=42,
    stratify=y_train_val
)

In [12]:
print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))

Training: 41801
Validation: 8960
Test: 8958


### Fit the TF-IDF vectorizer only on the training data, then transform the validation and test sets.

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.95
)

X_train_tfidf = tfidf.fit_transform(X_train)

X_val_tfidf = tfidf.transform(X_val)

X_test_tfidf = tfidf.transform(X_test)

### This avoids data leakage because the vocabulary is learned only from the training set.

## Save all the files

In [14]:
joblib.dump(tfidf, "../../models/tfidf_vectorizer.pkl")

joblib.dump(X_train_tfidf, "../../data/processed/X_train_tfidf.pkl")
joblib.dump(X_val_tfidf, "../../data/processed/X_val_tfidf.pkl")
joblib.dump(X_test_tfidf, "../../data/processed/X_test_tfidf.pkl")

joblib.dump(y_train, "../../data/processed/y_train.pkl")
joblib.dump(y_val, "../../data/processed/y_val.pkl")
joblib.dump(y_test, "../../data/processed/y_test.pkl")

['../../data/processed/y_test.pkl']

### Save the cleaned dataset

In [15]:
df.to_csv(
    "../../data/processed/amazon_reviews_sentiment_cleaned.csv",
    index=False
)

In [16]:
train_indices = X_train.index
val_indices = X_val.index
test_indices = X_test.index

joblib.dump(train_indices, "../../data/processed/train_indices.pkl")
joblib.dump(val_indices, "../../data/processed/val_indices.pkl")
joblib.dump(test_indices, "../../data/processed/test_indices.pkl")

['../../data/processed/test_indices.pkl']